# Aprendizaje automático interpretable para estratificar el riesgo de eventos adversos materno-perinatales

## Cuaderno reproducible para Google Colab

**Estudio:** Aprendizaje automático interpretable para estratificar el riesgo de eventos adversos materno-perinatales en La Guajira (Colombia), 2016–2022.

**Manuscrito:** AG-0.9-2026 — Revista Facultad Nacional de Salud Pública (Universidad de Antioquia).

**Propósito de este cuaderno:**

1. Permitir la **replicabilidad total** de los análisis presentados en el artículo.
2. Documentar **paso a paso** cada decisión metodológica conforme a las guías RECORD (estudios sobre datos rutinarios), TRIPOD-AI (modelos predictivos clínicos con inteligencia artificial) y CAIR (Clinical Artificial Intelligence Research, Olczak et al., 2021).
3. Servir como **material docente** para investigadores en epidemiología computacional y salud pública.

**¿Cómo se usa este cuaderno?**

1. Subir el archivo `dataset_MME_LaGuajira_2016_2022.xlsx` cuando el cuaderno lo solicite (celda de carga).
2. Ejecutar las celdas en orden (Menú → Entorno de ejecución → Ejecutar todo).
3. Las figuras y tablas se generan automáticamente y reproducen las del artículo.

**Tiempo aproximado de ejecución completa en Colab:** entre tres y cinco minutos.

---

**Marco causal del estudio** *(véase Figura 1 del manuscrito)*:

```
Determinantes distales  →  Mediadores estructurales  →  Complicaciones clínicas  →  MME severa
  edad, etnia, ruralidad     acceso, CPN efectivo,         hipertensión, hemorragia,    (desenlace OMS)
                              pertinencia intercultural     sepsis
```

> ⚠️ **Nota ética:** Los datos están anonimizados conforme a la Resolución 8430 de 1993 (Colombia) y la Ley 1581 de 2012 de protección de datos personales. El protocolo cuenta con aval del Comité Institucional de Ética en Investigación de la Universidad de La Guajira (acta n.° 04 del 12 de marzo de 2023).


## Paso 1. Instalación de dependencias

Google Colab trae preinstaladas la mayoría de librerías de ciencia de datos (numpy, pandas, scikit-learn, matplotlib, seaborn). Solo necesitamos asegurar las versiones de **xgboost**, **imbalanced-learn**, **statsmodels** y **openpyxl** (para leer Excel).

La línea con `!pip install` es un comando de shell (no Python) — Colab permite ejecutar comandos del sistema operativo prefijándolos con `!`.

In [ ]:
# Instalación silenciosa de paquetes (la opción -q oculta mensajes verbosos)
!pip install -q xgboost==2.1.4 imbalanced-learn==0.13.0 statsmodels==0.14.4 openpyxl==3.1.5

# Verificamos las versiones instaladas
import xgboost, imblearn, statsmodels, openpyxl, sklearn, pandas, numpy, matplotlib
print(f"xgboost:          {xgboost.__version__}")
print(f"imbalanced-learn: {imblearn.__version__}")
print(f"statsmodels:      {statsmodels.__version__}")
print(f"openpyxl:         {openpyxl.__version__}")
print(f"scikit-learn:     {sklearn.__version__}")
print(f"pandas:           {pandas.__version__}")
print(f"numpy:            {numpy.__version__}")
print(f"matplotlib:       {matplotlib.__version__}")


## Paso 2. Importación de librerías

Agrupamos las importaciones por funcionalidad para mejorar la legibilidad.

- **numpy y pandas**: manejo de arreglos numéricos y tablas de datos.
- **matplotlib y seaborn**: visualizaciones.
- **scikit-learn**: pipeline de ML clásico (regresión logística, árboles, KNN, naïve Bayes, SVM, random forest, gradient boosting), validación cruzada, métricas e importancia por permutación.
- **xgboost**: algoritmo de potenciación extrema por gradiente.
- **imblearn**: SMOTE (Synthetic Minority Over-sampling Technique) para abordar el desbalance de clases.
- **statsmodels**: regresión logística inferencial con razones de odds e intervalos de confianza al 95 %.

In [ ]:
# === Manipulación de datos ===
import numpy as np                           # álgebra lineal y operaciones vectorizadas
import pandas as pd                          # estructura DataFrame y operaciones sobre tablas
import warnings; warnings.filterwarnings('ignore')

# === Visualización ===
import matplotlib.pyplot as plt              # gráficos estáticos
import matplotlib as mpl                     # configuración global de estilo
import seaborn as sns                        # gráficos estadísticos sobre matplotlib

# === Machine Learning (scikit-learn) ===
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                              precision_score, recall_score, balanced_accuracy_score,
                              brier_score_loss, roc_curve, precision_recall_curve,
                              confusion_matrix, classification_report)

# === Gradient boosting de XGBoost ===
from xgboost import XGBClassifier

# === Manejo de desbalance ===
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline   # IMPORTANTE: usar el Pipeline de imblearn,
                                                          # no el de sklearn, para que SMOTE se aplique
                                                          # solo durante el entrenamiento.

# === Estadística clásica ===
import statsmodels.api as sm                # regresión logística con OR e IC 95 %
from scipy import stats                     # pruebas chi-cuadrado y Mann-Whitney

# === Carga de archivos en Colab ===
from google.colab import files              # diálogo para subir archivos al entorno

print("✓ Todas las librerías cargadas correctamente.")


## Paso 3. Configuración visual

Establecemos parámetros gráficos consistentes con los del artículo: tipografía serif (compatible con Times New Roman), tamaños legibles y resolución de 600 dpi. Definimos también la paleta **Okabe-Ito**, segura para personas con daltonismo y legible en impresión a escala de grises.

In [ ]:
# === Configuración tipográfica ===
mpl.rcParams['font.family'] = 'serif'                              # familia serif (estilo Times)
mpl.rcParams['font.size'] = 12                                     # tamaño base 12 pt
mpl.rcParams['axes.titlesize'] = 13
mpl.rcParams['axes.labelsize'] = 12
mpl.rcParams['xtick.labelsize'] = 11
mpl.rcParams['ytick.labelsize'] = 11
mpl.rcParams['legend.fontsize'] = 10

# === Apariencia limpia (sin bordes superiores/derechos) ===
mpl.rcParams['axes.spines.right'] = False
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.linewidth'] = 0.9

# === Resolución alta para publicación ===
mpl.rcParams['savefig.dpi'] = 600
mpl.rcParams['figure.dpi'] = 100   # en pantalla 100 dpi; al guardar se usan 600

# === Paleta Okabe-Ito (color-blind safe) ===
OK = {
    'azul':    '#0072B2',   # azul fuerte — algoritmo lineal
    'naranja': '#E69F00',   # naranja — XGBoost
    'verde':   '#009E73',   # verde — gradient boosting
    'amarillo':'#F0E442',
    'celeste': '#56B4E9',
    'rojo':    '#D55E00',   # bermellón — random forest (mejor modelo)
    'rosa':    '#CC79A7',
    'gris':    '#5A5A5A',
    'negro':   '#000000',
}

# === Semilla para reproducibilidad ===
SEMILLA = 42
np.random.seed(SEMILLA)

print("✓ Configuración visual y semilla aplicadas.")
print(f"  Semilla aleatoria fija: {SEMILLA}")


## Paso 4. Carga del dataset

Al ejecutar la siguiente celda se abrirá un diálogo de carga. Selecciona el archivo `dataset_MME_LaGuajira_2016_2022.xlsx` (el que descargaste junto a este cuaderno).

**El archivo tiene tres hojas:**
- `datos`: 5 706 registros × 13 columnas (12 predictores + 1 desenlace).
- `diccionario_variables`: descripción y valores válidos de cada variable.
- `metadata`: información sobre el origen, periodo, ética y población del estudio.

In [ ]:
# Diálogo interactivo de carga de archivo
print("Por favor, sube el archivo 'dataset_MME_LaGuajira_2016_2022.xlsx'")
print("(se abrirá un diálogo de selección de archivos)")
uploaded = files.upload()                              # devuelve un diccionario {nombre: bytes}

# Tomamos el primer archivo subido
nombre_archivo = next(iter(uploaded.keys()))
print(f"\n✓ Archivo subido: {nombre_archivo}")
print(f"  Tamaño: {len(uploaded[nombre_archivo])/1024:.1f} KB")


In [ ]:
# Leemos las tres hojas del Excel en DataFrames separados
df_datos       = pd.read_excel(nombre_archivo, sheet_name='datos')
df_diccionario = pd.read_excel(nombre_archivo, sheet_name='diccionario_variables')
df_metadata    = pd.read_excel(nombre_archivo, sheet_name='metadata')

print(f"Hoja 'datos':           {df_datos.shape[0]} filas × {df_datos.shape[1]} columnas")
print(f"Hoja 'diccionario':     {df_diccionario.shape[0]} variables documentadas")
print(f"Hoja 'metadata':        {df_metadata.shape[0]} campos de información del estudio")

print("\n=== Primeras 5 filas del dataset ===")
df_datos.head()


In [ ]:
# Mostramos el diccionario de variables para tener referencia rápida
df_diccionario


In [ ]:
# Mostramos los metadatos del estudio
df_metadata


## Paso 5. Análisis exploratorio (descriptiva univariada)

Antes de modelar revisamos:

1. **Estructura y tipos** de las variables.
2. **Valores faltantes** por columna.
3. **Distribución del desenlace** (¿está balanceado o desbalanceado?).
4. **Estadísticas descriptivas** de las variables continuas y categóricas.

Esto cumple con los ítems 13 (datos) y 14 (descriptivos) de la guía RECORD.

In [ ]:
# Información estructural del dataset
print("=== ESTRUCTURA ===")
df_datos.info()


In [ ]:
# Conteo de valores faltantes por columna
print("=== VALORES FALTANTES ===")
missing = df_datos.isna().sum()
missing_pct = (missing / len(df_datos) * 100).round(2)
pd.DataFrame({'faltantes_n': missing, 'faltantes_pct': missing_pct})


In [ ]:
# Distribución del desenlace
print("=== DISTRIBUCIÓN DE MME SEVERA (variable objetivo) ===")
conteo = df_datos['MME_severa'].value_counts().sort_index()
prevalencia = df_datos['MME_severa'].mean() * 100
print(f"  Casos negativos (0): {conteo.get(0, 0):>5}")
print(f"  Casos positivos (1): {conteo.get(1, 0):>5}")
print(f"  Prevalencia:         {prevalencia:.2f} %")
print(f"\n  ⚠️  El desenlace está marcadamente desbalanceado (clase positiva < 5 %).")
print(f"      Esto justifica el uso de SMOTE y de métricas como AUPRC además de AUC-ROC.")


In [ ]:
# Estadísticas descriptivas de las variables continuas
print("=== ESTADÍSTICAS DESCRIPTIVAS (continuas) ===")
continuas = ['edad_materna','numero_gestaciones','antecedente_abortos',
             'numero_controles_prenatales','semana_inicio_control_prenatal']
df_datos[continuas].describe().round(2)


## Paso 6. Preprocesamiento

**Decisiones explícitas (conforme a RECORD ítem 7 y CAIR ítem 'data preparation'):**

1. **Imputación**: rellenamos faltantes con la mediana en variables continuas y con la categoría modal en categóricas. La proporción de faltantes es baja (< 5 %), por lo que la imputación simple no introduce sesgo material.
2. **Estandarización**: variables continuas centradas en 0 con desviación 1, requerida por algoritmos basados en distancia (KNN, SVM) y regularización.
3. **Separación X/y**: `X` = matriz de predictores; `y` = vector binario del desenlace MME severa.

No aplicamos one-hot encoding porque las variables categóricas en Sivigila ya están codificadas numéricamente y los algoritmos de árboles tratan bien los códigos enteros.

In [ ]:
# Definimos predictores y desenlace
predictores = [c for c in df_datos.columns if c != 'MME_severa']
X = df_datos[predictores].copy()
y = df_datos['MME_severa'].astype(int)

# Imputación con mediana (estrategia conservadora para variables continuas y códigos)
for col in X.columns:
    if X[col].isna().any():
        X[col] = X[col].fillna(X[col].median())

print(f"X (predictores): {X.shape[0]} filas × {X.shape[1]} columnas")
print(f"y (desenlace):   {y.shape[0]} observaciones, {y.sum()} casos positivos")
print(f"\n✓ Sin valores faltantes después de la imputación: {X.isna().sum().sum() == 0}")


## Paso 7. Análisis bivariado (cada predictor vs. desenlace)

Para cada variable continua usamos la **prueba U de Mann-Whitney** (no asume normalidad). Para cada variable categórica usamos la **prueba chi-cuadrado de Pearson**. Reportamos el valor p sin corrección por múltiples comparaciones (exploratorio); la corrección formal queda para el análisis multivariado.

In [ ]:
# Función de bivariado continuo
def bivariado_continuo(df, var, target='MME_severa'):
    g0 = df[df[target]==0][var].dropna()
    g1 = df[df[target]==1][var].dropna()
    u, p = stats.mannwhitneyu(g0, g1, alternative='two-sided')
    return {'variable': var,
            'mediana_sin_MME_severa': g0.median(),
            'mediana_con_MME_severa': g1.median(),
            'p_valor': p}

# Función de bivariado categórico
def bivariado_categorico(df, var, target='MME_severa'):
    ct = pd.crosstab(df[var], df[target])
    chi2, p, _, _ = stats.chi2_contingency(ct)
    return {'variable': var,
            'mediana_sin_MME_severa': '—',
            'mediana_con_MME_severa': '—',
            'p_valor': p}

# Ejecutamos
filas = []
for v in ['edad_materna','numero_gestaciones','antecedente_abortos',
          'numero_controles_prenatales','semana_inicio_control_prenatal']:
    filas.append(bivariado_continuo(df_datos, v))
for v in ['area_residencia','pertenencia_etnica','grupo_indigena']:
    filas.append(bivariado_categorico(df_datos, v))

tab_bivariado = pd.DataFrame(filas)
tab_bivariado['significativo'] = tab_bivariado['p_valor'] < 0.05
print("=== ANÁLISIS BIVARIADO (MME severa) ===")
tab_bivariado.round(4)


## Paso 8. Regresión logística multivariada con razones de odds (OR) e IC 95 %

Esta es la **Figura 5 del artículo**: un modelo inferencial que estima asociaciones independientes entre determinantes categorizados y MME severa, ajustando por todas las variables simultáneamente.

**Categorizaciones aplicadas** (basadas en literatura clínica y el DAG del estudio):
- `edad_<20` y `edad_>=35`: extremos etarios de riesgo obstétrico.
- `rural`: residencia rural dispersa.
- `indigena`: pertenencia étnica indígena.
- `CPN_insuficiente`: menos de 4 controles prenatales.
- `inicio_CPN_tardio`: inicio del CPN después de la semana 16.
- `multiparidad`: 4 o más gestaciones.

In [ ]:
# Construimos variables categorizadas
df_log = df_datos.copy()
df_log['edad_<20']            = (df_log['edad_materna'] < 20).astype(int)
df_log['edad_>=35']           = (df_log['edad_materna'] >= 35).astype(int)
df_log['rural']               = (df_log['area_residencia'] == 3).astype(int)
df_log['indigena']            = (df_log['pertenencia_etnica'] == 1).astype(int)
df_log['CPN_insuficiente']    = (df_log['numero_controles_prenatales'] < 4).astype(int)
df_log['inicio_CPN_tardio']   = (df_log['semana_inicio_control_prenatal'] > 16).astype(int)
df_log['multiparidad']        = (df_log['numero_gestaciones'] >= 4).astype(int)

# Matriz de diseño y respuesta
cols_log = ['edad_<20','edad_>=35','rural','indigena',
            'CPN_insuficiente','inicio_CPN_tardio','multiparidad']
X_log = df_log[cols_log].astype(float)
y_log = df_log['MME_severa']

# Añadimos constante (intercepto) y ajustamos el modelo
X_log_sm = sm.add_constant(X_log)
modelo_logit = sm.Logit(y_log, X_log_sm).fit(disp=0, maxiter=200)

# Extraemos OR y sus IC 95 %
params = modelo_logit.params
conf   = modelo_logit.conf_int()
ps     = modelo_logit.pvalues
OR     = np.exp(params)
ORlo   = np.exp(conf[0])
ORhi   = np.exp(conf[1])

# Tabla resumen
tabla_OR = pd.DataFrame({
    'Variable':  params.index,
    'OR':        OR.values.round(3),
    'IC95_inf':  ORlo.values.round(3),
    'IC95_sup':  ORhi.values.round(3),
    'p_valor':   ps.values.round(3),
})
tabla_OR['significativo'] = tabla_OR['p_valor'] < 0.05
print("=== REGRESIÓN LOGÍSTICA MULTIVARIADA ===")
print(f"  N = {len(df_log)}, eventos = {int(y_log.sum())}, prevalencia = {y_log.mean()*100:.2f} %")
print(f"  Pseudo R²: {modelo_logit.prsquared:.4f}")
print(f"  Log-likelihood: {modelo_logit.llf:.2f}\n")
tabla_OR


## Paso 9. Forest plot de razones de odds (Figura 5 del artículo)

Visualizamos las OR ajustadas en escala logarítmica. Codificación cromática:
- **Rojo (rombo)**: factor de riesgo significativo (p < 0,05 y IC 95 % > 1).
- **Azul (cuadrado)**: protector significativo (p < 0,05 y IC 95 % < 1).
- **Gris (círculo)**: asociación no significativa.

In [ ]:
# Excluimos la constante para el forest plot
reg = tabla_OR[tabla_OR['Variable'] != 'const'].copy()
etiquetas = {
    'edad_<20':           'Edad materna < 20 años',
    'edad_>=35':          'Edad materna ≥ 35 años',
    'rural':              'Residencia rural dispersa',
    'indigena':           'Pertenencia étnica indígena',
    'CPN_insuficiente':   'Controles prenatales < 4',
    'inicio_CPN_tardio':  'Inicio CPN > 16 semanas',
    'multiparidad':       'Multiparidad ≥ 4 gestaciones',
}
reg['Etiqueta'] = reg['Variable'].map(etiquetas)
reg = reg.sort_values('OR')

fig, ax = plt.subplots(figsize=(11, 5.5))
y_pos = np.arange(len(reg))

# Bucle: graficamos cada fila con su color y marcador
for i, fila in enumerate(reg.itertuples()):
    if fila.p_valor < 0.05 and fila.IC95_inf > 1:
        c, mk, ls = OK['rojo'], 'D', '-'      # riesgo significativo
    elif fila.p_valor < 0.05 and fila.IC95_sup < 1:
        c, mk, ls = OK['azul'], 's', '-'      # protector significativo
    else:
        c, mk, ls = OK['gris'], 'o', '--'     # no significativo
    ax.plot([fila.IC95_inf, fila.IC95_sup], [i, i], color=c, linewidth=2.5, linestyle=ls, alpha=0.85)
    ax.scatter([fila.OR], [i], s=140, color=c, zorder=10, edgecolor='black', linewidth=1.1, marker=mk)
    sig = ' *' if fila.p_valor < 0.05 else ''
    ax.text(fila.IC95_sup*1.04, i,
            f'OR = {fila.OR:.2f} [{fila.IC95_inf:.2f}–{fila.IC95_sup:.2f}]; p = {fila.p_valor:.3f}{sig}',
            va='center', fontsize=10, color='#222')

# Línea vertical de no efecto (OR = 1)
ax.axvline(1, color='black', linestyle=':', linewidth=1.4, alpha=0.85)
ax.text(0.45, len(reg)+0.4, 'Protector',      ha='center', fontsize=10, color=OK['azul'], fontweight='bold')
ax.text(2.0,  len(reg)+0.4, 'Factor de riesgo', ha='center', fontsize=10, color=OK['rojo'], fontweight='bold')

# Leyenda manual
from matplotlib.lines import Line2D
leg = [
    Line2D([0],[0], marker='D', color='w', label='Factor de riesgo significativo (p < 0,05)',
           markerfacecolor=OK['rojo'], markersize=10, markeredgecolor='black'),
    Line2D([0],[0], marker='s', color='w', label='Protector significativo (p < 0,05)',
           markerfacecolor=OK['azul'], markersize=10, markeredgecolor='black'),
    Line2D([0],[0], marker='o', color='w', label='No significativo',
           markerfacecolor=OK['gris'], markersize=10, markeredgecolor='black'),
]
ax.legend(handles=leg, loc='lower right', fontsize=9.5, frameon=True, edgecolor='#999')

ax.set_yticks(y_pos); ax.set_yticklabels(reg['Etiqueta'], fontsize=11)
ax.set_xscale('log'); ax.set_xlim(0.3, 5.5)
ax.set_xlabel('Razón de odds ajustada (IC 95 %)  —  escala logarítmica', fontsize=11.5)
ax.grid(axis='x', alpha=0.25, linestyle=':', which='both')
ax.set_axisbelow(True); ax.set_ylim(-0.7, len(reg)+0.9)
plt.tight_layout()
plt.savefig('figura5_forest_OR.png', dpi=600, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Figura guardada: figura5_forest_OR.png")


## Paso 10. Comparación sistemática de ocho algoritmos de aprendizaje automático

Esta es la **Figura 2 del artículo**. Comparamos:

| Familia | Algoritmo | Razón |
|---|---|---|
| Lineal | Regresión logística | Baseline interpretable |
| Árbol único | Árbol de decisión | Baseline no lineal |
| Distancia | K-vecinos (k=15) | Baseline geométrico |
| Probabilístico | Naïve Bayes | Baseline bayesiano |
| Núcleo | Máquinas de vectores soporte RBF | Frontera no lineal |
| Ensamble | Random Forest | Bagging de árboles |
| Ensamble | Gradient Boosting | Boosting secuencial |
| Ensamble | XGBoost | Boosting optimizado |

**Validación**: cinco particiones estratificadas (preserva la prevalencia en cada fold).
**Manejo de desbalance**: SMOTE *solo en entrenamiento* dentro del pipeline (evita filtración de información).
**Métricas reportadas**: AUC-ROC, AUPRC, F1, sensibilidad, precisión positiva, exactitud balanceada y puntaje de Brier.

In [ ]:
# Definimos los ocho algoritmos
modelos = {
    'Reg. logística':    LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEMILLA),
    'Árbol decisión':    DecisionTreeClassifier(class_weight='balanced', random_state=SEMILLA, max_depth=6),
    'KNN (k=15)':        KNeighborsClassifier(n_neighbors=15),
    'Naive Bayes':       GaussianNB(),
    'SVM (RBF)':         SVC(probability=True, class_weight='balanced', random_state=SEMILLA),
    'Random Forest':     RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                                 random_state=SEMILLA, n_jobs=-1, max_depth=10),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=SEMILLA, max_depth=4),
    'XGBoost':           XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                                        scale_pos_weight=(y==0).sum()/(y==1).sum(),
                                        random_state=SEMILLA, use_label_encoder=False,
                                        eval_metric='auc', verbosity=0),
}

# Validación cruzada estratificada
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

# Bucle principal de evaluación
resultados = []
print("Entrenando 8 modelos × 5 folds. Esto puede tardar 1-3 minutos...\n")
for nombre, base in modelos.items():
    # Pipeline: estandarización → SMOTE (solo en entrenamiento) → modelo
    pipe = ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote',  SMOTE(random_state=SEMILLA, k_neighbors=5)),
        ('clf',    base),
    ])
    aucs, aps, f1s, recs, precs, baccs, briers = [], [], [], [], [], [], []
    for tr_idx, te_idx in skf.split(X, y):
        Xt, Xv = X.iloc[tr_idx], X.iloc[te_idx]
        yt, yv = y.iloc[tr_idx], y.iloc[te_idx]
        pipe.fit(Xt, yt)
        proba = pipe.predict_proba(Xv)[:, 1]
        pred  = (proba >= 0.5).astype(int)
        aucs.append(roc_auc_score(yv, proba))
        aps.append(average_precision_score(yv, proba))
        f1s.append(f1_score(yv, pred, zero_division=0))
        recs.append(recall_score(yv, pred, zero_division=0))
        precs.append(precision_score(yv, pred, zero_division=0))
        baccs.append(balanced_accuracy_score(yv, pred))
        briers.append(brier_score_loss(yv, proba))
    resultados.append({
        'Modelo':         nombre,
        'AUC_media':      np.mean(aucs),
        'AUC_sd':         np.std(aucs),
        'AUC_ic95_inf':   np.mean(aucs) - 1.96*np.std(aucs)/np.sqrt(5),
        'AUC_ic95_sup':   np.mean(aucs) + 1.96*np.std(aucs)/np.sqrt(5),
        'AUPRC':          np.mean(aps),
        'F1':             np.mean(f1s),
        'Sensibilidad':   np.mean(recs),
        'Precisión':      np.mean(precs),
        'Exact_balanc':   np.mean(baccs),
        'Brier':          np.mean(briers),
    })
    print(f"  {nombre:22s}  AUC = {np.mean(aucs):.3f} ± {np.std(aucs):.3f}  AUPRC = {np.mean(aps):.3f}")

# Tabla final ordenada por AUC
tabla_modelos = pd.DataFrame(resultados).sort_values('AUC_media', ascending=False).reset_index(drop=True)
print("\n=== TABLA RESUMEN ===")
tabla_modelos.round(3)


## Paso 11. Figura 2: Comparación de AUC entre los ocho algoritmos

In [ ]:
# Ordenamos ascendentemente para que el mejor quede arriba
res_ord = tabla_modelos.sort_values('AUC_media', ascending=True).reset_index(drop=True)
maxauc = res_ord['AUC_media'].max()
top3_idx = res_ord['AUC_media'].nlargest(3).index.tolist()

fig, ax = plt.subplots(figsize=(9, 5.8))
y_pos = np.arange(len(res_ord))
colors = []
for i, auc in enumerate(res_ord['AUC_media']):
    if auc == maxauc:       colors.append(OK['rojo'])
    elif i in top3_idx:     colors.append(OK['naranja'])
    else:                   colors.append(OK['azul'])

# Barras de error (IC 95 %)
ax.errorbar(res_ord['AUC_media'], y_pos,
            xerr=[res_ord['AUC_media']-res_ord['AUC_ic95_inf'],
                  res_ord['AUC_ic95_sup']-res_ord['AUC_media']],
            fmt='none', ecolor='#444', capsize=5, capthick=1.2, linewidth=1.2, zorder=2)
for i, (auc, c) in enumerate(zip(res_ord['AUC_media'], colors)):
    ax.scatter([auc], [i], s=180, color=c, zorder=10, edgecolor='black', linewidth=1.2, marker='o')
for i, auc in enumerate(res_ord['AUC_media']):
    ax.text(auc + 0.005, i + 0.18, f'{auc:.3f}', fontsize=9.5, fontweight='bold', color='#222')

ax.axvline(0.5, color='#888', linestyle=':',  linewidth=1.4, label='Azar (AUC = 0,50)')
ax.axvline(0.7, color=OK['verde'], linestyle='--', linewidth=1.4, label='Umbral aceptable (AUC = 0,70)')

# Resalta el mejor
best_i = res_ord['AUC_media'].idxmax()
ax.axhspan(best_i - 0.4, best_i + 0.4, color=OK['rojo'], alpha=0.08, zorder=0)

ax.set_yticks(y_pos); ax.set_yticklabels(res_ord['Modelo'], fontsize=11)
ax.set_xlabel('Área bajo la curva ROC (validación cruzada 5-fold, IC 95 %)', fontsize=12)
ax.set_xlim(0.55, 0.82)
ax.legend(loc='lower right', frameon=True, edgecolor='#888', fontsize=10)
ax.grid(axis='x', alpha=0.25, linestyle=':')
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('figura2_AUC_comparacion.png', dpi=600, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Figura guardada: figura2_AUC_comparacion.png")


## Paso 12. Importancia de variables por permutación (Figura 3 del artículo)

Entrenamos el **modelo ganador** (Random Forest) sobre todo el dataset y calculamos para cada variable cuánto disminuye el AUC cuando barajamos sus valores. Una caída grande implica que esa variable es informativa.

Repetimos la permutación 10 veces por variable para obtener un estimador estable con desviación estándar.

In [ ]:
# Entrenamos el pipeline Random Forest sobre todo el dataset
pipe_rf = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote',  SMOTE(random_state=SEMILLA, k_neighbors=5)),
    ('clf',    RandomForestClassifier(n_estimators=500, class_weight='balanced',
                                       random_state=SEMILLA, n_jobs=-1, max_depth=10)),
])
pipe_rf.fit(X, y)

print("Calculando importancia por permutación (10 repeticiones × 12 variables)...")
perm = permutation_importance(pipe_rf, X, y, n_repeats=10,
                              random_state=SEMILLA, n_jobs=-1, scoring='roc_auc')

imp_df = pd.DataFrame({
    'Variable':           X.columns,
    'Importancia_media':  perm.importances_mean,
    'Importancia_sd':     perm.importances_std,
}).sort_values('Importancia_media', ascending=True)
print("\n=== IMPORTANCIA DE VARIABLES (Random Forest) ===")
imp_df.sort_values('Importancia_media', ascending=False).round(4)


In [ ]:
# Figura 3: barras horizontales con paleta viridis
fig, ax = plt.subplots(figsize=(9, 6))
norm = imp_df['Importancia_media'] / imp_df['Importancia_media'].max()
colors_v = plt.cm.viridis(0.15 + 0.7 * norm)
ax.barh(imp_df['Variable'], imp_df['Importancia_media'],
        xerr=imp_df['Importancia_sd'],
        color=colors_v, edgecolor='black', linewidth=0.7,
        error_kw={'elinewidth': 1.2, 'capsize': 4, 'capthick': 1, 'ecolor':'#333'})
for i, (val, sd) in enumerate(zip(imp_df['Importancia_media'], imp_df['Importancia_sd'])):
    if val > 0.0001:
        ax.text(val + sd + 0.005, i, f'{val:.3f} ± {sd:.3f}',
                va='center', fontsize=9.5, color='#222', fontweight='bold')
ax.set_xlabel('Reducción de AUC al permutar la variable (importancia ± DE)', fontsize=11.5)
ax.axvline(0, color='black', linewidth=0.7)
ax.set_xlim(-0.005, imp_df['Importancia_media'].max() * 1.45)
ax.grid(axis='x', alpha=0.25, linestyle=':')
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('figura3_importancia.png', dpi=600, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Figura guardada: figura3_importancia.png")


## Paso 13. Curvas ROC y precisión–recall de los cuatro mejores (Figura 4)

Acumulamos predicciones de los cinco folds por modelo y trazamos las curvas. Para que sean legibles en color y en blanco y negro usamos **codificación redundante**: cada modelo lleva un color único, un estilo de línea único y aparece etiquetado en la leyenda con su métrica.

In [ ]:
# Modelos de interés
modelos_top = {
    'Reg. logística':    LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEMILLA),
    'Random Forest':     RandomForestClassifier(n_estimators=500, class_weight='balanced',
                                                 random_state=SEMILLA, n_jobs=-1, max_depth=10),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=SEMILLA, max_depth=4),
    'XGBoost':           XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                                        scale_pos_weight=(y==0).sum()/(y==1).sum(),
                                        random_state=SEMILLA, use_label_encoder=False,
                                        eval_metric='auc', verbosity=0),
}

# Acumulamos predicciones de cada fold
all_y = {n:[] for n in modelos_top}
all_p = {n:[] for n in modelos_top}
for tr_idx, te_idx in skf.split(X, y):
    Xt, Xv = X.iloc[tr_idx], X.iloc[te_idx]
    yt, yv = y.iloc[tr_idx], y.iloc[te_idx]
    for nombre, base in modelos_top.items():
        pipe = ImbPipeline([
            ('s',  StandardScaler()),
            ('sm', SMOTE(random_state=SEMILLA, k_neighbors=5)),
            ('c',  base),
        ])
        pipe.fit(Xt, yt)
        proba = pipe.predict_proba(Xv)[:, 1]
        all_y[nombre].extend(yv.tolist())
        all_p[nombre].extend(proba.tolist())

# Codificación redundante: color + estilo de línea + marcador
estilos = [
    ('Reg. logística',    OK['azul'],    '--', 's'),
    ('Gradient Boosting', OK['verde'],   '-.', '^'),
    ('XGBoost',           OK['naranja'], ':',  'D'),
    ('Random Forest',     OK['rojo'],    '-',  'o'),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
# Panel A: ROC
for n, c, ls, mk in estilos:
    fpr, tpr, _ = roc_curve(all_y[n], all_p[n])
    a = roc_auc_score(all_y[n], all_p[n])
    axes[0].plot(fpr, tpr, color=c, linewidth=2.4, linestyle=ls,
                  label=f"{n} (AUC = {a:.3f})", alpha=0.95)
axes[0].plot([0,1],[0,1], color='#666', linestyle=':', linewidth=1.3, label='Azar')
axes[0].set_xlabel('1 − Especificidad', fontsize=12)
axes[0].set_ylabel('Sensibilidad',      fontsize=12)
axes[0].set_title('A. Curva ROC', fontsize=13, fontweight='bold', loc='left')
axes[0].legend(loc='lower right', fontsize=10, frameon=True, edgecolor='#999')
axes[0].grid(alpha=0.25, linestyle=':')
axes[0].set_aspect('equal'); axes[0].set_xlim(0,1); axes[0].set_ylim(0,1.02)

# Panel B: Precision-Recall
for n, c, ls, mk in estilos:
    pre, rec, _ = precision_recall_curve(all_y[n], all_p[n])
    ap = average_precision_score(all_y[n], all_p[n])
    axes[1].plot(rec, pre, color=c, linewidth=2.4, linestyle=ls,
                  label=f"{n} (AUPRC = {ap:.3f})", alpha=0.95)
axes[1].axhline(y.mean(), color='#444', linestyle=':', linewidth=1.5,
                label=f'Prevalencia ({y.mean()*100:.1f} %)')
axes[1].set_xlabel('Sensibilidad (Recall)',         fontsize=12)
axes[1].set_ylabel('Valor predictivo positivo',     fontsize=12)
axes[1].set_title('B. Curva Precisión–Recall', fontsize=13, fontweight='bold', loc='left')
axes[1].legend(loc='upper right', fontsize=10, frameon=True, edgecolor='#999')
axes[1].grid(alpha=0.25, linestyle=':')
axes[1].set_xlim(0,1); axes[1].set_ylim(0, 0.6)

plt.tight_layout()
plt.savefig('figura4_ROC_PR.png', dpi=600, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Figura guardada: figura4_ROC_PR.png")


## Paso 14. Matriz de confusión del modelo ganador (Random Forest)

Reportamos los aciertos y errores del modelo en un esquema 2×2: verdaderos positivos (VP), falsos positivos (FP), verdaderos negativos (VN) y falsos negativos (FN), con el punto de corte estándar 0,5.

In [ ]:
# Predicción binaria por validación cruzada agrupada
mejor = 'Random Forest'
y_real = np.array(all_y[mejor])
y_pred = (np.array(all_p[mejor]) >= 0.5).astype(int)
cm = confusion_matrix(y_real, y_pred)

# Visualizamos la matriz
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No MME severa', 'MME severa'],
            yticklabels=['No MME severa', 'MME severa'],
            ax=ax, cbar=False, annot_kws={'fontsize': 14, 'fontweight': 'bold'})
ax.set_xlabel('Predicción del modelo', fontsize=12)
ax.set_ylabel('Realidad observada',    fontsize=12)
ax.set_title(f'Matriz de confusión — {mejor}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('matriz_confusion_RF.png', dpi=600, bbox_inches='tight', facecolor='white')
plt.show()

# Reporte detallado
print(f"\n=== REPORTE DE CLASIFICACIÓN — {mejor} ===")
print(classification_report(y_real, y_pred, target_names=['No MME severa','MME severa']))


## Paso 15. Descargar las figuras generadas

Cuando termine la ejecución, Colab guarda los archivos en el entorno virtual. La siguiente celda los empaqueta y los descarga al equipo del usuario.

In [ ]:
import zipfile
archivos = ['figura2_AUC_comparacion.png','figura3_importancia.png',
            'figura4_ROC_PR.png','figura5_forest_OR.png','matriz_confusion_RF.png']

with zipfile.ZipFile('figuras_resultados.zip', 'w') as zf:
    for f in archivos:
        try: zf.write(f)
        except FileNotFoundError: print(f"⚠️  No se encontró {f}")

files.download('figuras_resultados.zip')
print("✓ Archivo 'figuras_resultados.zip' descargado.")


## Conclusión del cuaderno

Has reproducido íntegramente los análisis cuantitativos del manuscrito:

| Resultado | Valor esperado | Figura/Tabla del artículo |
|---|---|---|
| Tamaño efectivo de la cohorte de MME | 5 706 | Tabla 1 |
| Prevalencia de MME severa | ≈ 4,5 % | Tabla 1 |
| AUC del mejor modelo (Random Forest) | ≈ 0,73 (IC 0,69–0,77) | Figura 2 |
| Variable más informativa | Tipo de terminación de la gestación | Figura 3 |
| AUPRC del Random Forest | ≈ 0,13 | Figura 4 |
| OR pertenencia indígena | 1,72 (IC 1,26–2,34) | Figura 5 |
| OR controles prenatales insuficientes | 1,48 (IC 1,12–1,94) | Figura 5 |

**Si los valores coinciden, la replicación fue exitosa.**

### Próximos pasos sugeridos

1. Probar diferentes puntos de corte para optimizar sensibilidad o precisión según el caso de uso clínico.
2. Estratificar por subgrupos étnicos para análisis de equidad algorítmica (fairness).
3. Validar el modelo en una cohorte temporal (entrenar 2016–2020, validar 2021–2022).
4. Integrar variables clínicas adicionales (presión arterial, hemoglobina, comorbilidades) cuando estén disponibles.

### Documentos de transparencia que acompañan al artículo

- **Checklist RECORD diligenciado** (`checklist_RECORD_diligenciado.docx`).
- **Checklist CAIR diligenciado** (`checklist_CAIR_diligenciado.docx`) — referencia: Olczak J, *et al.* Acta Orthop. 2021;92(5):513-525.
- **Manuscrito final** (AG-0.9-2026).

---

*Cuaderno preparado conforme a los lineamientos de la Revista Facultad Nacional de Salud Pública (Universidad de Antioquia) y a las guías STROBE, RECORD, TRIPOD-AI, CAIR y SRQR.*
